# Module 4: Agentic and Multi-Agent Systems Engineering

This notebook is refactored into small blocks so each stage of the workflow can be run and debugged independently.


## 1. Setup and Dependencies

Install runtime dependencies if needed (safe to skip if already installed in your environment).


In [1]:
%pip install -q langgraph langchain langchain-openai langchain-community qdrant-client pypdf networkx rank_bm25


Note: you may need to restart the kernel to use updated packages.


a:\PROJECTS\obsidi_AI_academy\.venv\Scripts\python.exe: No module named pip


## 2. Imports and Path Setup

Import core libraries and configure access to `snapaudit-integrated` for `PolicyRetriever`.


In [2]:
import sys
import os
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional, TypedDict, Annotated
import operator

from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage, AIMessage

module_dir = Path.cwd()
integrated_dir = (module_dir / "../snapaudit-integrated").resolve()
sys.path.append(str(integrated_dir))

print(f"✓ Integrated module path: {integrated_dir}")


✓ Integrated module path: A:\PROJECTS\obsidi_AI_academy\snapaudit-integrated


### Import PolicyRetriever

Try to import the production retriever. If unavailable, we will fall back to a mock implementation.


In [3]:
try:
    from policy_retriever import PolicyRetriever, PolicyVerdict
    HAS_POLICY_RETRIEVER = True
    print("✓ Successfully imported PolicyRetriever")
except ImportError as e:
    HAS_POLICY_RETRIEVER = False
    print(f"⚠ Could not import PolicyRetriever: {e}")


⚠ Could not import PolicyRetriever: No module named 'policy_retriever'


## 3. Initialize Policy Engine

Initialize the real policy retriever when possible, otherwise use a deterministic mock for graph testing.


In [4]:
class MockPolicyRetriever:
    def check_compliance(self, receipt_data: Dict[str, Any], role: str = "Employee"):
        total = receipt_data.get("total", 0)
        approved = total < 100
        return type('obj', (object,), {
            'approved': approved,
            'reason': 'Mock reason: amount threshold',
            'citations': ['Mock citation']
        })


def init_policy_retriever():
    if not HAS_POLICY_RETRIEVER:
        print("⚠ Using MockPolicyRetriever (import unavailable)")
        return MockPolicyRetriever()

    policy_pdf_path = integrated_dir / "sample_expense_policy.pdf"
    if not policy_pdf_path.exists():
        print(f"⚠ PDF not found at {policy_pdf_path}, using mock")
        return MockPolicyRetriever()

    try:
        retriever = PolicyRetriever(str(policy_pdf_path))
        print("✓ PolicyRetriever initialized")
        return retriever
    except Exception as e:
        print(f"⚠ Failed to initialize PolicyRetriever ({e}), using mock")
        return MockPolicyRetriever()


policy_retriever = init_policy_retriever()


⚠ Using MockPolicyRetriever (import unavailable)


## 4. Tool Layer

Define tool functions the agent can call for policy checks, notifications, and receipt lookup.


In [5]:
MOCK_RECEIPTS = {
    "R001": {"id": "R001", "total": 20.00, "category": "Lunch", "merchant": "Sandwich Shop", "date": "2024-01-15"},
    "R002": {"id": "R002", "total": 200.00, "category": "Client Dinner", "merchant": "Fancy Steakhouse", "date": "2024-01-16"},
    "R003": {"id": "R003", "total": 5000.00, "category": "Office Supplies", "merchant": "Apple Store", "date": "2024-01-17"},
}


def get_receipt_tool(receipt_id: str) -> Dict[str, Any]:
    """Fetch receipt details from the database."""
    return MOCK_RECEIPTS.get(receipt_id, {})


def check_policy_tool(receipt_data: Dict[str, Any], role: str = "Employee") -> Dict[str, Any]:
    """Check if a receipt complies with policy."""
    print(f"--- [Tool: CheckPolicy] Checking {receipt_data.get('category')} for ${receipt_data.get('total')} ---")
    verdict = policy_retriever.check_compliance(receipt_data, role)
    return {
        "approved": verdict.approved,
        "reason": verdict.reason,
        "citations": verdict.citations,
    }


def send_email_tool(to_email: str, subject: str, body: str) -> str:
    """Simulate sending an email request for clarification."""
    print(f"--- [Tool: SendEmail] To: {to_email} | Subject: {subject} ---")
    return f"Email sent to {to_email}"


print("✓ Tools defined")


✓ Tools defined


## 5. State Schema

Define the graph state shared across planner, executor, and critic nodes.


In [6]:
class AuditState(TypedDict):
    receipt_id: str
    receipt_data: Dict[str, Any]
    verdict: Optional[Dict[str, Any]]
    status: str  # pending, approved, denied, flagged_for_human, error
    messages: Annotated[List[BaseMessage], operator.add]
    next_step: str


## 6. Node Definitions

Implement the three-agent loop:
- Planner: decide next action
- Executor: call tools
- Critic: assign final status


In [7]:
def planner_node(state: AuditState):
    print("--- [Node: Planner] ---")
    receipt_id = state["receipt_id"]

    if not state.get("receipt_data"):
        print(f"Fetching receipt {receipt_id}...")
        receipt = get_receipt_tool(receipt_id)
        return {"receipt_data": receipt, "next_step": "check_policy"}

    if not state.get("verdict"):
        return {"next_step": "check_policy"}

    return {"next_step": "critic"}


def executor_node(state: AuditState):
    print("--- [Node: Executor] ---")
    step = state["next_step"]

    if step == "check_policy":
        receipt = state["receipt_data"]
        verdict = check_policy_tool(receipt)
        return {"verdict": verdict}

    return {}


def critic_node(state: AuditState):
    print("--- [Node: Critic] ---")
    verdict = state.get("verdict")
    if not verdict:
        return {"status": "error", "messages": [AIMessage(content="No verdict found.")]}

    receipt_total = state["receipt_data"].get("total", 0)
    if verdict["approved"]:
        if receipt_total > 1000:
            return {"status": "flagged_for_human", "messages": [AIMessage(content="High value approval requires human review.")]}
        return {"status": "approved", "messages": [AIMessage(content="Receipt approved automatically.")]}

    return {"status": "denied", "messages": [AIMessage(content=f"Receipt denied: {verdict['reason']}")]}


## 7. Graph Construction

Wire nodes and conditional routes into a cyclic LangGraph workflow.


In [8]:
workflow = StateGraph(AuditState)

workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("critic", critic_node)

workflow.set_entry_point("planner")


def route_planner(state):
    if state.get("next_step") == "check_policy":
        return "executor"
    if state.get("next_step") == "critic":
        return "critic"
    return END


def route_executor(state):
    return "planner"


def route_critic(state):
    status = state.get("status")
    if status in ["approved", "denied", "flagged_for_human", "error"]:
        return END
    return "planner"


workflow.add_conditional_edges("planner", route_planner)
workflow.add_conditional_edges("executor", route_executor)
workflow.add_conditional_edges("critic", route_critic)

app = workflow.compile()
print("✓ Graph compiled")


✓ Graph compiled


## 8. Execution Tests

Validate low-risk and high-risk receipts.


In [9]:
def run_audit(receipt_id: str):
    initial_state: AuditState = {
        "receipt_id": receipt_id,
        "messages": [],
        "status": "pending",
        "receipt_data": {},
        "verdict": None,
        "next_step": "",
    }
    return app.invoke(initial_state)


### Test 1: Simple Lunch (Expected: approved)


In [10]:
result_r001 = run_audit("R001")
print("Final Status:", result_r001["status"])
print("Verdict:", result_r001.get("verdict"))


--- [Node: Planner] ---
Fetching receipt R001...
--- [Node: Executor] ---
--- [Tool: CheckPolicy] Checking Lunch for $20.0 ---
--- [Node: Planner] ---
--- [Node: Critic] ---
Final Status: approved
Verdict: {'approved': True, 'reason': 'Mock reason: amount threshold', 'citations': ['Mock citation']}


### Test 2: High Value Expense (Expected: flagged_for_human)


In [11]:
result_r003 = run_audit("R003")
print("Final Status:", result_r003["status"])
print("Verdict:", result_r003.get("verdict"))


--- [Node: Planner] ---
Fetching receipt R003...
--- [Node: Executor] ---
--- [Tool: CheckPolicy] Checking Office Supplies for $5000.0 ---
--- [Node: Planner] ---
--- [Node: Critic] ---
Final Status: denied
Verdict: {'approved': False, 'reason': 'Mock reason: amount threshold', 'citations': ['Mock citation']}


## 9. What to Build Next

Add an explicit Human-Approval node and persistence so audits can pause/resume after manual review.
